In [1]:
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_core.runnables import RunnableLambda,RunnableWithMessageHistory
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_community.chat_message_histories import RedisChatMessageHistory



c:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loaded_docs=TextLoader("demo.txt",encoding="utf-8")
docs=loaded_docs.load()

In [3]:
from langchain_experimental.text_splitter import SemanticChunker

In [4]:
# embeddings=OpenAIEmbeddings(model="text-embedding-3-small")
# chunker=SemanticChunker(embeddings)

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n","\n"," ",""]
)
chunks=text_splitter.split_documents(docs)

In [6]:
for i,chunk in enumerate(chunks):
    print(f" chunk : {i+1} \n{chunk.page_content} \n\n")

 chunk : 1 
Langchain is a framework for buliding applications using LLMs.

Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.


You can create chains,memory , agents and retrievers.


The Eiffel tower is located in Paris.


France is a Popular tourist destination.


We can create agents using langchain.


OpenAI develops AI models such as GPT-4o, GPT-4.1, and embedding models used in retrieval systems. 


 chunk : 2 
A common RAG pipeline includes loading data, splitting text into chunks, creating embeddings, and storing vectors in a database.


When a user asks a question, the retriever finds top-k relevant chunks and sends them to the language model as context.


Good chunking strategy improves retrieval quality because important facts stay together in semantically meaningful units. 


 chunk : 3 
History-aware retrieval rewrites follow-up questions into standalone queries, for example converting "How does it help?" into "How does RAG help?"

In [7]:
embeddings=OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore=FAISS.from_documents(chunks,embeddings)

In [8]:
retriever=vectorstore.as_retriever(search_kwargs={"k":3})

In [9]:
condense_prompt=ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="chat_history"),
    ("human","{input}"),
    ("human","Based on the above input write a standalon search query. If the query iteself sufficient written it as it is")
])

from langchain_classic.chains import create_history_aware_retriever

In [10]:
llm=ChatOpenAI(model="gpt-3.5-turbo")
history_aware_retriever=create_history_aware_retriever(llm,retriever,condense_prompt)

In [11]:
rag_prompt=ChatPromptTemplate.from_messages([
    ("system","""
You are a helpful assistant and you will have to answer user's query from the followinf context : \n {context}
     If you don't know the answer simply say you don't have the information
"""),
MessagesPlaceholder(variable_name="chat_history"),
("human","{input}")
])

In [12]:
# ----- OLD (in-memory) history approach: kept for reference -----
# store = {}
# def get_session_history(session_id: str):
#     if session_id not in store:
#         store[session_id] = InMemoryChatMessageHistory()
#     return store[session_id]

# ----- NEW (Redis) history approach -----
# Redis runs in Docker at localhost:6379 and stores chat messages by session_id.
# This survives notebook re-runs and is closer to production behavior.
redis_url = "redis://localhost:6379/0"

def get_session_history(session_id: str):
    return RedisChatMessageHistory(
        session_id=session_id,
        url=redis_url,
    )

In [13]:
def format_docs(docs):
    return  "\n\n".join(doc.page_content for doc in docs)

In [14]:
rag_chain=(
    {
        "context":history_aware_retriever|format_docs,
        "input":RunnableLambda(lambda x:x["input"]),
        "chat_history":RunnableLambda(lambda x : x["chat_history"])

    }
    |rag_prompt
    |llm
    |StrOutputParser()
)

In [15]:
rag_chain_with_history = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

In [ ]:
# 3) Use with session_id (no manual chat_history needed now)
# Keep same session_id to continue one conversation thread in Redis.
session_id = "user-2"

a1 = rag_chain_with_history.invoke(
    {"input": "What is LangChain?"},
    config={"configurable": {"session_id": session_id}},
)
print("A1:", a1)

a2 = rag_chain_with_history.invoke(
    {"input": "Does it support agents too?"},
    config={"configurable": {"session_id": session_id}},
)
print("A2:", a2)

# Try changing session_id = "user-2" to start a fresh chat memory.

A2: Yes, Langchain does support creating agents.


In [17]:
rag_chain.invoke({
    "input":"What is langchain?",
    "chat_history":[]
})

'Langchain is a framework for building applications using Large Language Models (LLMs). It provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone. With Langchain, you can create chains, memory, agents, and retrievers to work with LLMs and enhance their capabilities. Let me know if you would like more information on Langchain.'